In [0]:
loan_info=spark.read.format("delta").load("abfss://silver@deassessment.dfs.core.windows.net/loans/")
loan_info.createOrReplaceTempView("loan_info")


In [0]:
%sql
select * from loan_info

In [0]:
payment_transaction=spark.read.format("delta").load("abfss://silver@deassessment.dfs.core.windows.net/payment/")
payment_transaction.createOrReplaceTempView("payment_transaction")

In [0]:
%sql
select * from payment_transaction

In [0]:
%sql
select max(timestamp) from payment_transaction

In [0]:
%sql describe payment_transaction

In [0]:
df_transaction=spark.sql(f"""with extracted_components as (
    select 
        pt.amount as amount,
        pt.loan_id as loan_id,
        pt.metadata as metadata,
        pt.payment_id as payment_id,
        pt.payment_method as payment_method,
        pt.timestamp as timestamp,
        li.origination_date as origination_date,
        li.customer_id as customer_id,
        li.product_type as product_type,
        li.principal_amount as principal_amount,
        EXTRACT(YEAR FROM CAST(pt.timestamp AS TIMESTAMP)) AS target_year,
        EXTRACT(MONTH FROM CAST(pt.timestamp AS TIMESTAMP)) AS target_month,
        EXTRACT(DAY FROM CAST(li.origination_date AS DATE)) AS orig_day
    from payment_transaction pt
    inner join loan_info li on pt.loan_id = li.loan_id
    where li.status in ('active', 'default')
),

rollover_logic as (
    select
        amount,
        loan_id,
        metadata,
        payment_id,
        payment_method,
        timestamp,
        origination_date,
        customer_id,
        product_type,
        target_year,
        target_month,
        orig_day,
        principal_amount,
        -- Apply the calendar logic explicitly 
        CASE 
            WHEN target_month = 2 AND orig_day > 28 THEN 1
            WHEN target_month IN (4, 6, 9, 11) AND orig_day = 31 THEN 1
            ELSE orig_day 
        END AS final_day,
        
        CASE 
            WHEN target_month = 2 AND orig_day > 28 THEN 3 
            WHEN target_month IN (4, 6, 9, 11) AND orig_day = 31 THEN target_month + 1 
            ELSE target_month
        END AS final_month
    from extracted_components
)

select 
    amount,
    loan_id,
    metadata,
    payment_id,
    payment_method,
    timestamp,
    customer_id,
    product_type,
    origination_date,
    principal_amount,
    make_date(target_year, final_month, final_day) AS due_date 
from rollover_logic""")

df_transaction.createOrReplaceTempView("df_transaction")

In [0]:
df_transaction.display()
df_transaction.printSchema()

In [0]:
from pyspark.sql.functions import col, when, datediff, current_date, to_date, lit, sum, round

# 1. Cast string columns to actual Date types for accurate comparison
df_dates = df_transaction.withColumn("due_date_parsed", col("due_date").cast("date")) \
             .withColumn("payment_date_parsed", to_date(col("timestamp")))

# 2. Apply the dynamic DPD conditional logic
df_with_dpd = df_dates.withColumn(
    "days_past_due",
    when(col("payment_date_parsed").isNull(), datediff(current_date(), col("due_date_parsed"))) \
    .when(col("payment_date_parsed") > col("due_date_parsed"), datediff(col("payment_date_parsed"), col("due_date_parsed"))) \
    .otherwise(lit(0))
)

In [0]:
df_with_dpd.display()

In [0]:
df_final = df_with_dpd.groupBy("product_type").agg(
    # Part A: Total active portfolio volume
    sum("principal_amount").alias("total_active_balance"),
    
    # Part B: Isolate ONLY the balances where DPD is between 30 and 59
    sum(when(col("days_past_due").between(30, 59), col("amount")).otherwise(0)).alias("delinquent_balance_30d")
).withColumn(
    # Part C: Compute the final rate percentage
    "delinquency_rate_pct",
    round((col("delinquent_balance_30d") / col("total_active_balance")) * 100, 5)
)

In [0]:
df_final.display()

In [0]:
df_final.write.mode('overwrite').parquet("abfss://gold@deassessment.dfs.core.windows.net/delinquency_rate_30d_by_loan_product")